## **Pandas Cleaning Checklist** 

**Goal:** Detect and correct (or remove) corrupt, inaccurate, incomplete, irrelevant, or improperly formatted records to improve data quality for subsequent analysis

### **Table of Contents**
* Profiling
* Column Fixes
* Filtering Data
* Missing Data
* Duplicate Data
* Correct Data Types
* Numeric Data
* Text Data
* Date and Time Data
* Verify

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [26]:
dataset = pd.read_csv("C:/Users/rcdan/Documents/Career Dev/GitHub Repos/project-portfolio/Python/Cancer_Study/Cancer_Incidents_Extract.csv")
dataset.head()


,Source.Name,Name,Domain,Indicator,Year,GeogID,Race_Ethnicity,Gender,Age_Group,Month,Measure,Value,ts,measureName,indicatorName,contentAreaName,Race_Ethnicitylabel,Genderlabel
0,bladder_male.xlsx,ARIZONA,CR,18,2015-2019,ALL,ALL,m,ALL,ALL,221,30.4,2023-10-04T09:15:11.880,Age-adjusted incidence rate of Bladder Cancer ...,Incidence of Bladder Cancer,Cancer,Any Race Ethnicity,Male
1,bladder_male.xlsx,YUMA,CR,18,2015-2019,4027,ALL,m,ALL,ALL,221,24.4,2023-10-04T09:15:11.880,Age-adjusted incidence rate of Bladder Cancer ...,Incidence of Bladder Cancer,Cancer,Any Race Ethnicity,Male
2,bladder_male.xlsx,YAVAPAI,CR,18,2015-2019,4025,ALL,m,ALL,ALL,221,38.4,2023-10-04T09:15:11.880,Age-adjusted incidence rate of Bladder Cancer ...,Incidence of Bladder Cancer,Cancer,Any Race Ethnicity,Male
3,bladder_male.xlsx,SANTA CRUZ,CR,18,2015-2019,4023,ALL,m,ALL,ALL,221,24.9,2023-10-04T09:15:11.880,Age-adjusted incidence rate of Bladder Cancer ...,Incidence of Bladder Cancer,Cancer,Any Race Ethnicity,Male
4,bladder_male.xlsx,PINAL,CR,18,2015-2019,4021,ALL,m,ALL,ALL,221,30.4,2023-10-04T09:15:11.880,Age-adjusted incidence rate of Bladder Cancer ...,Incidence of Bladder Cancer,Cancer,Any Race Ethnicity,Male


### **Profiling**

#### Summaries

In [ ]:
# Display the first few rows
print("First 5 rows:\n", dataset.head())

# Random Sample
print("\nSample of Data:\n", dataset.sample(n=5))

# Get a summary of the DataFrame, including data types and non-null counts
print("\nDataFrame Info:\n", dataset.info())

# Get descriptive statistics for numerical columns
print("\nNumerical Statistics:\n", dataset.describe())

#### Column Distributions

In [ ]:
# Create a list of the specific columns for distributions
columns_for_dist = ['Name', 'GeogID', 'measureName']
# Loop through the columns in your list
for col in columns_for_dist:
    print(f"\nValue Counts for {col}:\n", dataset[col].value_counts(dropna=False))

### **Column Fixes** 

#### Remove Columns

In [ ]:
# Drop columns manually 
dataset = dataset.drop(columns=['Source.Name', 'Domain', 'Indicator', 'Year', 'GeogID', 'Race_Ethnicity', 'Gender', 'Age_Group', 'Month', 'Measure', 'ts', 'measureName', 'contentAreaName', 'Race_Ethnicitylabel'])

# Remove columns with too many missing values (e.g., > 5% missing)
missing_percentage = (dataset.isna().sum() / len(dataset)) * 100
cols_to_drop = missing_percentage[missing_percentage > 5].index
dataset = dataset.drop(cols_to_drop, axis=1)

#### Rename Columns

In [ ]:
# Rename columns
dataset = dataset.rename(columns={'Name': 'county', 'Value': 'cancer_rate', 'indicatorName': 'cancer_type'})

#### Move Column

In [ ]:
# move ['ID'] to first column
cols_to_move = ['ID']
remaining_cols = [col for col in dataset.columns if col not in cols_to_move]
new_order = cols_to_move + remaining_cols # or switch to put column last 
dataset = dataset[new_order]

### **Filtering**

#### Filter by Operators

In [ ]:
# Filter By Operators
# YUMA & rate > 10
dataset = dataset[(dataset['Name'] == 'YUMA') & (dataset['Value'] > 10)]

# YUMA or NAVAJO
dataset = dataset[(dataset['Name'] != 'YUMA') | (dataset['GenderLabel'] == 'Male')]

# rate > 10 and NOT (YUMA, NAVAJO, or mohave) 
dataset = dataset[(dataset['Value'] > 10) & 
                  ~(dataset['Name'].isin(['YUMA','NAVAJO','MOHAVE']))]

#### Filter by String Values

In [ ]:
# na=False NaN rows will be kept in the DataFrame, but their value in the filter mask = False; best practice
#.strip() method helps produce more robust filtering with .contains

# kidney string
dataset = dataset[dataset['indicatorName'].str.strip().str.contains('Kidney', case=False, na=False)] 

# Kidney or Bladder Cancer string
dataset = dataset[dataset['indicatorName'].str.strip().str.contains('Kidney|Bladder', case=False, na=False)] 

# Starts with 'b' and does not end with 'A'
dataset = dataset[(dataset['Source.Name'].str.strip().str.startswith('b',na=False)) & 
              ~(dataset['Name'].str.strip().str.endswith('A',na=False))]

### **Missing Values**

#### Identify Non-Standard Missing Values First

In [ ]:
# Replace with NaN examples: empty strings '', N/A, hyphen, 0 as a placeholder, 999
dataset = dataset.replace('', np.nan)

# Replace with NaT examples: empty strings '', N/A, hyphen, 0 as a placeholder, 999
dataset['date_col'] = dataset['date_col'].replace('', np.nan)
dataset['date_col'] = pd.to_datetime(dataset['date_col'])

#### Identify Standard Missing Values: NaN, None, NaT

In [ ]:
# Identify standard missing values. Will not identify non-standard missing values. 
print("Missing Values:\n", dataset.isna().sum())

#### Imputation

In [ ]:
# Impute numerical missing values with the mean or median
dataset['numerical_column'] = dataset['numerical_column'].fillna(dataset['numerical_column'].mean())

# Impute string missing values with a custom string
dataset['Genderlabel'] = dataset['Genderlabel'].fillna('single gender')

#### Forward/Backward Fill

In [ ]:
# Forward fill missing values (use with caution, especially for time series)
dataset['Genderlabel'] = dataset['Genderlabel'].ffill()

# Backward fill missing values (use with caution)
dataset['Genderlabel'] = dataset['Genderlabel'].bfill()

#### Flag Column with Missing Values

In [ ]:
# Create boolean flag column of missing values
dataset['gender_missing'] = dataset['Genderlabel'].isna()

# Create binary flag column of missing values
dataset['gender_missing'] = dataset['Genderlabel'].isna().astype(int)

#### Remove Rows

In [ ]:
# Remove entire rows with any missing values 
dataset = dataset.dropna()

# Remove entire rows if certain columns have missing values
dataset = dataset.dropna(subset='Genderlabel')

# Remove entire rows where ALL cells in the row are NaN
dataset = dataset.dropna(how='all')

### **Duplicate Data**

#### Identify amount of duplicate rows

In [ ]:
# Identify amount of duplicate rows where all columns form a duplicate
print("Number of duplicate rows:", dataset.duplicated().sum())

# Identify amount of duplicate rows where a subset of columns form a duplicate
columns_forming_duplicate = ['measureName', 'indicatorName']
print("Number of duplicate rows:", dataset.duplicated(subset=['measureName', 'indicatorName']).sum())

#### View duplicate rows

In [ ]:
# keep= 'first' [default] to keep the first occurrence of the record, marking the rest as dupes
print("\nDuplicate Rows:\n", dataset[dataset.duplicated(keep='first')]) 

# keep= 'last' to keep the last occurrence of the record, marking the rest as dupes
print("\nDuplicate Rows:\n", dataset[dataset.duplicated(keep='last')]) 

# keep= False identifies ALL occurrences of a duplicate set 
print("\nDuplicate Rows:\n", dataset[dataset.duplicated(keep=False)]) 

# View duplicate rows where where a subset of columns form a duplicate
print("\nDuplicate Rows:\n", dataset[dataset.duplicated(subset=['measureName', 'indicatorName'],keep='first')]) 

#### Remove Duplicate Rows

In [ ]:
# Remove duplicate rows, keeping the first occurrence
dataset = dataset.drop_duplicates(keep='first')

# Remove duplicate rows, keeping the last occurrence
dataset = dataset.drop_duplicates(keep='last')

# Identify and keep only truly unique rows (where no exact duplicate exists anywhere)
# keep=False identifies ALL occurrences of a duplicate set 
dataset = dataset.drop_duplicates(keep=False)

# Remove duplicates based on a subset of columns
dataset = dataset.drop_duplicates(subset=['measureName', 'indicatorName'], keep='first')

### **Correct Data Types**

#### View Data types

In [ ]:
# Check current data types only 
print("\nCurrent Data Types:\n", dataset.dtypes)

# Get a summary of the DataFrame, including data types and non-null counts
print("\nDataFrame Info:\n", dataset.info())

#### Convert Multiple Columns

In [ ]:
# Change multiple
dataset = dataset.astype({'Value': 'float64','Measure': 'int64', 'Gender': 'category','GeogID': 'string'})
dataset.dtypes

#### Convert To numeric

In [ ]:
# Convert a column to int64 or float64
dataset['numerical_column'] = pd.to_numeric(dataset['numerical_column'], errors='coerce') # 'coerce' will turn errors into NaN

# Remove commas and convert to int64 or float 64
dataset = dataset['thousands_column'].str.replace(',', '') 
dataset['thousands_column'] = pd.to_numeric(dataset['thousands_column'], errors = 'coerce')

#### Convert To datetime

In [ ]:
# # Convert to datetime64 
dataset['ts'] = pd.to_datetime(dataset['ts'], format='%Y-%m-%d', errors='coerce')

# Truncate datetime to have time component 00:00:00
dataset['date_column_normalized'] = dataset['date_column'].dt.normalize()

#### Convert to date object

In [ ]:
# pandas does not have designated date type
dataset['ts'] = dataset['ts'].str[:10] # extract date component as string
dataset['ts'] = pd.to_datetime(dataset['ts'], format='%Y-%m-%d', errors='coerce') # convert string to datetime64
dataset['ts'] = dataset['ts'].dt.date # convert datetime64 back to object type to remove time component

#### Convert to categorical or boolean

In [ ]:
# Convert pre-defined buckets (pd.cut) into categorical for more Memory Efficiency, Performance Improvement, Statistical Signaling, Optional Defined Order (Ordinal Data)
dataset['categorical_column'] = dataset['categorical_column'].astype('category') 

# why boolean? supports logic operations, pandas' boolean identifies missing values instead of Numpy coercing them to T/F
# Convert 0/1 to boolean
dataset['binary_column'] = dataset['binary_column'].astype('boolean') 

# Map string variations to T/F
dataset['adverse_effects'] = dataset['adverse_effects'].str.lower().str.strip().map({'no': False, 'yes':True}).astype('boolean') 

#### Convert to String

In [ ]:
# StringDtype > 'str' for performance and without converting NaN's to strings
dataset['id_column'] = dataset['id_column'].astype('string') 

### **Numeric Data**

#### Absolute Value

In [ ]:
# Negative Data that shouldn't be negative
dataset['Value'] =  dataset['Value'].abs() 

#### Replace entire cell value, regardless of data type

In [ ]:
# Replace entire cell value, regardless of data type
dataset['country_column'] = dataset['country_column'].replace([00000, 'zero'], 0)

#### Identify Quartiles & Outliers

In [ ]:
# For numerical columns, using IQR (Interquartile Range) method
Q1 = dataset['numerical_column'].quantile(0.25)
Q3 = dataset['numerical_column'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = dataset[(dataset['numerical_column'] < lower_bound) | (dataset['numerical_column'] > upper_bound)]
print("\nOutliers in numerical_column:\n", outliers)

sns.catplot(kind='box', data=dataset, y='numerical_column') 
plt.title('numerical_column')
plt.show()

#### Handle Outliers

In [ ]:
# Option 1: Remove outliers/ data that falls outside of a desired min & max
dataset_no_outliers = dataset[~((dataset['numerical_column'] < lower_bound) | (dataset['numerical_column'] > upper_bound))]

# Option 2: Cap or floor outliers, replacing outliers with the lower/upper bound value
dataset['numerical_column_capped'] = dataset['numerical_column'].clip(lower=lower_bound, upper=upper_bound)

### **Text Data**

#### Extract Text Relative to Position 

In [ ]:
# Extract N left characters
dataset['new_column'] = dataset['string_column'].str[:N]

# Extract N right characters
dataset['new_column'] = dataset['string_column'].str[-N:]

# Extract all characters from position N
dataset['new_column'] = dataset['new_column'].str[N-1:]

# Extract Substring from character 3 and go to 5 characters
dataset['new_column'] = dataset['string_column'].str[2:7]

#### Split String to Columns

In [ ]:
# Split the 'Name' column into 'First Name' and 'Last Name'
dataset[['First Name', 'Last Name']] = dataset['Name'].str.split(',', expand=True)

#### Create Composite Primary Key

In [ ]:
# Example creating primary key with split, extract, and combine

# extract cancer name and gender code
dataset['cancer_name'] = dataset['cancer_type'].str.split(' ').str[2]
dataset['gender_code'] = dataset['gender'].str[0]
# create unique composite key of county, cancer_name, and gender_code
dataset['ID'] = (dataset['county'] + '-' + dataset['cancer_name'] + '-' + dataset['gender_code']).str.upper()
# remove intermediate columns
dataset = dataset.drop(columns=['cancer_name', 'gender_code'])


#### Extract Text between Delimiters 

In [ ]:
# create series object of split text
dataset['cancer_type'] = dataset['indicatorName'].str.split(' ')
# gets 3rd to the end elements of series and joins them with space
dataset['cancer_type'] = dataset['cancer_type'].str[2:].str.join(' ')

#### Remove whitespace or Special Characters

In [ ]:
# Remove leading/trailing whitespace
dataset['text_column'] = dataset['text_column'].str.strip()

# Remove special characters
# Replace anything that is NOT(^) in the set('[ ]') of alphanumeric & whitespace(\s) characters with ''
# handle set without escape sequence(r) and with Regular Expression commands(regex=True)
dataset['text_column'] = dataset['text_column'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

#### Replace cell value or substring

In [ ]:
# Replace entire cell value, regardless of data type
dataset['country_column'] = dataset['country_column'].replace(['U.S.A.', 1], 'USA')

# Replace substring within cell
dataset['text_column'] = dataset['text_column'].str.replace('misspelled', 'correct')

#### Convert Case

In [ ]:
# Convert case
dataset['text_column'] = dataset['text_column'].str.lower()
dataset['text_column'] = dataset['text_column'].str.upper()
dataset['text_column'] = dataset['text_column'].str.title()

### **Date & Time Data**

#### Extract Date components

In [ ]:
# Extract date components
dataset['year'] = dataset['date_column'].dt.year
dataset['month'] = dataset['date_column'].dt.month
dataset['day'] = dataset['date_column'].dt.day
dataset['day_of_week'] = dataset['date_column'].dt.day_name()

#### Extract time components

In [ ]:
# Extract time components if present
dataset['hour'] = dataset['datetime_column'].dt.hour
dataset['minute'] = dataset['datetime_column'].dt.minute
dataset['second'] = dataset['datetime_column'].dt.second

#### Extract Current date or timestamp

In [ ]:
# current date
current_date = pd.Timestamp.now().date()

# current Timestamp
current_timestamp = pd.Timestamp.now()

#### Convert time zones 

In [ ]:
# Handle time zones 
dataset['datetime_utc'] = dataset['datetime_column'].dt.tz_localize('UTC')
dataset['datetime_local'] = dataset['datetime_utc'].dt.tz_convert('US/Pacific')

### **Verify**

In [ ]:
# Re-run descriptive statistics and info
print("\nCleaned DataFrame Info:\n", dataset.info())
print("\nCleaned Numerical Statistics:\n", dataset.describe())

# Check for remaining missing values or duplicates
print("\nMissing Values After Cleaning:\n", dataset.isna().sum())
print("\nNumber of Duplicate Rows After Cleaning:", dataset.duplicated().sum())

# Sample the cleaned data to visually inspect
print("\nSample of Cleaned Data:\n", dataset.sample(5))